# Swiss roll: layerwise goal-oriented adaptivity

Reference implementation for

> M. Hintermüller, M. Hinze and D. Korolev,
> *Layerwise goal-oriented adaptivity for neural ODEs: an optimal control perspective*,
> Journal of Numerical Mathematics (2026).
> [doi:10.1515/jnma-2026-0008](https://doi.org/10.1515/jnma-2026-0008)

Binary classification of two concentric spirals (Section 8.1). This is the controlled setting in which to watch the local indicators respond to layer insertion and re-optimization, and to see how the regularization parameter $\lambda$ governs the stability of the adaptive hierarchy.

All numerics live in the `lgoa` package; this notebook only chooses parameters,
runs the solver and draws the figures. The two algorithms of the paper are

* **Algorithm 1** &mdash; `lgoa.solver.solve_fixed_depth`: full-batch $H^1$-Adam
  warm start, then BFGS in $H^1$-whitened coordinates, on a fixed temporal grid;
* **Algorithm 2** &mdash; `lgoa.adaptive.run`: solve &rarr; estimate $\eta_k$
  &rarr; mark &rarr; bisect &rarr; prolongate, until the layer budget is spent.

## 1. Install

In [ ]:
# On Colab this installs the package from GitHub. A local clone is used as-is.
try:
    import lgoa
except ImportError:
    !pip install -q "git+https://github.com/REPLACE-ME/layerwise-adaptive-neural-odes.git"
    import lgoa

import lgoa
from lgoa import plots

print(lgoa.backend_report())

## 2. Choose the experiment

The bulk of the hyperparameters (network size, Adam/BFGS budgets, early
stopping) lives in the YAML preset printed below &mdash; edit it only if you
want to depart from the published setup. The parameters worth varying from a
notebook are collected here.

In [ ]:
# ----------------------------------------------------------------- main knobs
MARKING                = "conservative"  # "conservative" or "dorfler"
DOERFLER_FRACTION      = 0.5           # bulk fraction, used when MARKING == "dorfler"
STATIONARITY_TOLERANCE = 1e-3          # eps_stat of Algorithm 1
REGULARIZATION_LAMBDA  = 1e-3          # lambda of the H^1 regularizer
INITIAL_DEPTH          = 2             # K_0
MAX_DEPTH              = 12            # K_max, the outer stopping rule
TRANSFER_ADAM_STATE    = True          # prolongate the Adam moments on refinement

# A reduced profile that finishes in a few minutes on a Colab CPU.
# Set to False for the setup reported in the paper (considerably slower).
QUICK = True

QUICK_PROFILE = dict(
    samples_per_class=400,     # paper: 1200
    n_candidates=3,            # paper: 10 initializations
    search_max_epochs=200,     # paper: 1000
    search_min_epochs=60,
    adam_max_steps=4000,       # paper: 20000
    bfgs_max_iterations=500,   # paper: 10000
    indicator_sup_samples=20,  # paper: 50
    max_depth=9,               # paper: 12
)
# ---------------------------------------------------------------------------

overrides = dict(
    marking=MARKING,
    dorfler_fraction=DOERFLER_FRACTION,
    stationarity_tolerance=STATIONARITY_TOLERANCE,
    reg_lambda=REGULARIZATION_LAMBDA,
    initial_depth=INITIAL_DEPTH,
    max_depth=MAX_DEPTH,
    transfer_adam_state=TRANSFER_ADAM_STATE,
)
if QUICK:
    overrides.update(QUICK_PROFILE)

cfg = lgoa.load_preset("swiss_roll", **overrides)
print(cfg.summary())
print("\npreset file:", lgoa.config_path("swiss_roll"))

## 3. Implementation checks

Six independent checks: the tridiagonal solve against SciPy, the hand-assembled
control and endpoint derivatives against `jax.grad`, invariance of the frozen
endpoint maps, the closed-form indicator pairing against dense quadrature, and
the equality of Dörfler refinement with one marked interval and plain
single-interval insertion.

In [ ]:
lgoa.self_check()

## 4. Data

In [ ]:
data = lgoa.load_dataset(cfg.problem)
print(data.summary())
plots.dataset(data);

## 5. Run the adaptive hierarchy

The endpoint maps are calibrated once on the initial partition and then frozen,
so every refinement round changes only *where* layers are inserted.

In [ ]:
history = lgoa.run(cfg, data)

In [ ]:
print(history.summary())

columns = [
    "phase", "depth", "objective", "data_loss", "validation_loss",
    "validation_accuracy", "stationarity_norm", "stationarity_status",
    "marked_intervals", "marked_indicator_share", "adam_steps", "bfgs_iterations",
]
history.phase_table[columns].round(6)

## 6. Figures

### 6.1 Quantities along the hierarchy

Stationarity measure against $\varepsilon_{\mathrm{stat}}$, the optimized
objective, both losses, and accuracy &mdash; each against the depth $K$.

In [ ]:
plots.learning_curves(history);

### 6.2 The adaptive grids

One row per phase, with the nodes marked; the triangles show where the next
layers are about to be inserted.

In [ ]:
plots.grid_history(history);

### 6.3 Refinement events

Local indicators $\eta_k$ before insertion, immediately after insertion, and
after re-optimization. Red bars are the marked parents on the left and their
children in the other two columns. A children-to-parent ratio below one means
re-optimization removed indicator mass from the refined region.

In [ ]:
plots.refinement_events(history, n_events=4);

### 6.4 Indicator components

How the adjoint, control and state contributions to $\eta_K$ evolve with depth,
and the local distribution on the final grid.

In [ ]:
plots.indicator_components(history);

### 6.5 State, control and adjoint

The triplet on the final grid. Weak regularization gives visibly more
oscillatory trajectories, i.e. a richer feature transformation.

In [ ]:
plots.trajectories(history, data);

### 6.6 Prediction of the trained neural ODE

In [ ]:
plots.decision_boundary(history, data);

## 7. Export

Figures as PDF and PNG, tables as CSV. On Colab the files appear in the file
browser on the left; the last line zips them for download.

In [ ]:
figure_paths = plots.save_all(history, data, "figures")
table_paths = history.to_csv("results")
print(f"{len(figure_paths)} figure files, {len(table_paths)} tables")

# Colab only: uncomment to download everything as a single archive.
# import shutil; from google.colab import files
# shutil.make_archive("lgoa_output", "zip", ".", "figures")
# files.download("lgoa_output.zip")

## 8. Conservative against Dörfler marking

Re-running from the same calibration keeps the comparison paired: identical
data, identical endpoint maps, identical initial control and the same layer
budget &mdash; only the marked set differs. Conservative marking inserts one
layer per round and so re-optimizes after every single insertion; Dörfler
marking inserts a whole bulk at once, which buys fewer rounds of global
re-optimization at the price of less selective placement.

```python
import pandas as pd

calibration = history.calibration          # reuse the frozen endpoint maps
runs = {"conservative": cfg.with_overrides(marking="conservative")}
for fraction in (0.25, 0.5, 0.75):
    runs[f"dorfler {fraction}"] = cfg.with_overrides(
        marking="dorfler", dorfler_fraction=fraction
    )

results = {
    name: lgoa.run(config, data, calibration, progress=False)
    for name, config in runs.items()
}
pd.DataFrame([r.totals() for r in results.values()], index=list(results))
```

Table 1 of the paper reports that $\lambda = 10^{-3}$ is the most robust compromise: stronger regularization stabilizes the hierarchy but limits expressivity, while $\lambda \le 10^{-5}$ makes stationarity progressively harder to reach. Re-run section 2 with `REGULARIZATION_LAMBDA = 1e-2` or `1e-6` to see both failure modes.